# Duplicate-Message (`_dup`) Handling in the Kerchunk Reference Generator

Demonstrates the behavior added for #259: when two GRIB2 messages carry fully
identical identifying metadata (same discipline, PDT octets, level, and lead
time), the generator emits the colliding messages as `_dup` variables instead
of silently overwriting the earlier message's chunk references.

The example uses a public NBM core file from NOMADS. NBM `PTYPE` output
carries four surface messages whose Sections 0/1/3/4 are byte-identical; they
differ only in their Section 5 packing and their data.

Requires this PR's branch installed (`pip install -e .`) plus `xarray`,
`fsspec`, and `zarr`.

In [ ]:
import datetime
import urllib.request
from pathlib import Path

# NOMADS retains roughly the last two days of NBM output; default to yesterday 00z.
date = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=1)).strftime("%Y%m%d")
url = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/blend/prod/"
       f"blend.{date}/00/core/blend.t00z.core.f001.ak.grib2")

grib_file = Path("blend.t00z.core.f001.ak.grib2")
if not grib_file.exists():
    print("downloading", url)
    urllib.request.urlretrieve(url, grib_file)
print(grib_file, f"{grib_file.stat().st_size/1e6:.1f} MB")

## Generate a manifest filtered to PTYPE

The filter keeps the example focused; the same handling applies to an
unfiltered manifest.

In [ ]:
import warnings
from grib2io.kerchunk import ReferenceGenerator

gen = ReferenceGenerator(str(grib_file), filters={"shortName": "PTYPE"})
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    gen.to_json("blend_ptype.json")

for w in caught:
    print("WARNING:", w.message)

One warning per collision round: the first message in file order keeps the
base name `PTYPE`; each later message whose dimension-value tuple matches an
already-emitted variable is re-emitted under the next suffix.

In [ ]:
import json

refs = json.load(open("blend_ptype.json"))["refs"]
variables = sorted({k.split("/")[0] for k in refs
                    if k.endswith("/.zarray") and k.count("/") == 1})
print(variables)

## Open the manifest and read every variant

All four messages remain reachable. On `main`, only one of these would exist
in the manifest and the other three messages' bytes would not be referenced
anywhere.

In [ ]:
import fsspec
import numpy as np
import xarray as xr
import grib2io.codecs  # registers the codec

fs = fsspec.filesystem("reference", fo="blend_ptype.json", skip_instance_cache=True)
ds = xr.open_dataset(fs.get_mapper(""), engine="zarr",
                     backend_kwargs={"consolidated": False})

for name in ["PTYPE", "PTYPE_dup", "PTYPE_dup2", "PTYPE_dup3"]:
    vals = np.asarray(ds[name].values).ravel()
    finite = vals[np.isfinite(vals)]
    uniq, counts = np.unique(finite, return_counts=True)
    top = ", ".join(f"{int(u)}: {c:,}" for u, c in
                    sorted(zip(uniq, counts), key=lambda t: -t[1])[:4])
    print(f"{name:11s} finite {finite.size:,}/{vals.size:,}  values -> {top}")

Each variant holds different data (different precipitation-type category
fields encoded under identical identifying metadata), confirming that keeping
only one of them would discard real, distinct fields.

## Why `_dup` rather than choosing or raising

Nothing in the identifying metadata distinguishes these messages, so the
generator cannot know which one a consumer intends. Raising would fail whole
files over one oddly-encoded product; skipping or overwriting is silent data
loss. Emitting every colliding message with a warning preserves the file's
full contents and leaves the choice with the consumer.

Many collisions in operational output differ in a PDT field the generator
does not yet track as a dimension (probability thresholds, categorical
values). Treating those as first-class dimensions would fold such messages
into proper arrays and reserve `_dup` for the truly identical residue — noted
as a follow-on in #259.